# Part VI: Applications: solutions to the code exercises

Worked solutions to the code exercises from [*Matrices in Practice*](https://trgoodale.github.io/matrix-book/appendices/solutions-6.html), by Tom Goodale.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trgoodale/matrix-book/blob/main/code/python/solutions-part-6.ipynb)

Run the cells in order, from the top: later cells use variables set in earlier ones. Try each exercise yourself first. Change the numbers and run them again to see what happens. The code needs NumPy, SciPy and Matplotlib (and pandas in a few chapters), all of which Colab already has.

Code licensed under the [MIT licence](https://trgoodale.github.io/matrix-book/LICENSE.md).

In [ ]:
# The data files are in the book's repository. Use the local copy if there is one,
# otherwise read them from GitHub (as in Colab).
import os
DATA = "../../data/" if os.path.exists("../../data/iris.csv") else "https://raw.githubusercontent.com/trgoodale/matrix-book/main/data/"

## Chapter 31: Structures: pin-jointed trusses

### Exercise 7

For the Pratt bridge, move a 20 kN load to each of the three interior joints of the bottom chord in turn. For each member, find the largest tension and the largest compression it ever carries. Which members must be designed for both?

In [ ]:
import numpy as np

def truss(P, members, supports, loads):
    P = np.array(P, dtype=float)
    reactions = [(j, d) for j, kind in supports.items() for d in ([0, 1] if kind == "pin" else [1])]
    A = np.zeros((2 * len(P), len(members) + len(reactions)))
    for k, (i, j) in enumerate(members):
        e = (P[j] - P[i]) / np.linalg.norm(P[j] - P[i])
        A[2 * i:2 * i + 2, k] += e
        A[2 * j:2 * j + 2, k] -= e
    for r, (j, d) in enumerate(reactions):
        A[2 * j + d, len(members) + r] = 1
    b = np.zeros(2 * len(P))
    for j, (fx, fy) in loads.items():
        b[2 * j:2 * j + 2] = [-fx, -fy]
    return A, b

joints = [[0, 0], [4, 0], [8, 0], [12, 0], [16, 0], [4, 4], [8, 4], [12, 4]]
members = [(0, 1), (1, 2), (2, 3), (3, 4), (5, 6), (6, 7), (1, 5), (2, 6),
           (3, 7), (0, 5), (4, 7), (5, 2), (7, 2)]
forces = []
for j in [1, 2, 3]:
    A, b = truss(joints, members, {0: "pin", 4: "roller"}, {j: (0, -20)})
    forces.append(np.linalg.solve(A, b)[:len(members)])
forces = np.array(forces)
for (i, j), col in zip(members, forces.T):
    print(f"{i}-{j}: max tension {max(col.max(), 0):6.2f}, max compression {abs(min(col.min(), 0)):6.2f}")

Over the three load positions, the bottom chord carries 5 to 15 kN of tension, the top chord 10 to 20 kN of compression, and the end posts 7.07 to 21.21 kN of compression. The two interior diagonals are the only members that change sign: each carries 14.14 kN of tension when the load is in the middle, or under it, but 7.07 kN of *compression* when the load is on the far side. They must be designed for both, which for a slender member in compression means guarding against buckling. The verticals at joints 1 and 3 carry 20 kN of tension when the load is directly on them, and the middle vertical never carries anything for these loads.

## Chapter 32: Electrical circuits

### Exercise 6

For the filter of Example 3, find the angular frequency at which $\lvert V_2/V_{\text{in}}\rvert = 1/\sqrt{2}$, by solving the complex system at many frequencies or with a root finder.

In [ ]:
import numpy as np
from scipy.optimize import brentq

R, C = 1e3, 1e-6
def gain(w):
    Y = np.array([[2 / R + 1j * w * C, -1 / R], [-1 / R, 1 / R + 1j * w * C]])
    return abs(np.linalg.solve(Y, [1 / R, 0])[1])

w3 = brentq(lambda w: gain(w) - 1 / np.sqrt(2), 1, 1e4)
print(w3, w3 / (2 * np.pi))

In MATLAB, `fzero(@(w) gain(w) - 1/sqrt(2), [1 1e4])` does the same, with `gain` defined as a function that builds and solves the same complex system. The output falls to $1/\sqrt{2}$ of the input (the **cut-off** or $-3$ dB point) at $\omega \approx 374$ rad/s, about 60 Hz: well below $1/(RC) = 1000$ rad/s, because the second stage loads the first.

## Chapter 33: Heat flow and finite differences

### Exercise 6

Solve the plate problem with the top at 100 °C and the other edges at 0 °C for $n = 10, 20, 40, 80$, with a sparse direct solver and with CG. Record the time taken and the number of CG steps. How does the number of CG steps grow with $n$? Plot the temperature along the vertical centre line for $n = 80$.

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve, cg
import time

for n in [10, 20, 40, 80]:
    T = sp.diags([-1.0, 2.0, -1.0], [-1, 0, 1], shape=(n, n))
    A = (sp.kron(sp.identity(n), T) + sp.kron(T, sp.identity(n))).tocsr()
    b = np.zeros((n, n)); b[0, :] = 100
    t0 = time.perf_counter(); u = spsolve(A, b.ravel()); t1 = time.perf_counter()
    steps = []
    cg(A, b.ravel(), rtol=1e-8, callback=lambda xk: steps.append(1)); t2 = time.perf_counter()
    print(f"n = {n:3d}: direct {t1 - t0:.4f} s, CG {t2 - t1:.4f} s in {len(steps)} steps")

The number of CG steps roughly doubles each time $n$ doubles, so it grows like $n$, the square root of the condition number (which grows like $n^2$). Timings vary between computers, but at these small sizes the direct solver is as fast as CG or faster (in some versions of MATLAB, `pcg` is much slower than backslash at $n = 80$). CG pulls ahead only for much larger problems, especially three-dimensional ones, where direct elimination fills in badly. Along the centre line, the temperature falls from 100 °C at the hot edge to 0 at the opposite edge, steeply near the hot edge and gently further away: by the middle of the plate it is already down to 25 °C, as Example 3 showed.

## Chapter 34: Computer graphics and robotics

### Exercise 6

For the two-link arm of Example 4, plot the hand's position as $\theta_1$ runs from $0°$ to $360°$ with $\theta_2$ fixed at $60°$, and again for $\theta_2 = 120°$. Then plot the region reached when both angles vary, and compare with the annulus drawn in the app.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

L1, L2 = 1.5, 1.0
t1 = np.radians(np.arange(0, 361))
fig, ax = plt.subplots(figsize=(5, 5))
for t2deg in [60, 120]:
    t2 = np.radians(t2deg)
    x = L1 * np.cos(t1) + L2 * np.cos(t1 + t2)
    y = L1 * np.sin(t1) + L2 * np.sin(t1 + t2)
    ax.plot(x, y, label=f"θ₂ = {t2deg}°")
T1, T2 = np.meshgrid(t1[::4], t1[::4])
ax.scatter(L1 * np.cos(T1) + L2 * np.cos(T1 + T2), L1 * np.sin(T1) + L2 * np.sin(T1 + T2), s=1, color="0.7")
ax.set_aspect("equal"); ax.legend()
plt.show()

With $\theta_2$ fixed, the hand moves on a circle about the base, of radius $\sqrt{L_1^2 + L_2^2 + 2L_1L_2\cos\theta_2}$: about 2.18 for $60°$ and 1.32 for $120°$. Letting both angles vary fills the annulus between radii $L_1 - L_2 = 0.5$ and $L_1 + L_2 = 2.5$, as drawn in the app. MATLAB code follows the same steps, with `cosd` and `sind`.

## Chapter 35: Markov chains and PageRank

### Exercise 6

Write a PageRank function that handles dead ends, by treating a page with no links as linking to every page. Apply it to the [**Dead end**](https://trgoodale.github.io/matrix-book/apps/random-surfer.html?preset=Dead%20end) web of the app (A links to B and C; B to C; C to nothing; D to A and C; E to D and F; F to D and E), and check your results against the app.

In [ ]:
import numpy as np

def pagerank(links, d=0.85, steps=200):
    """links[j] lists the pages that page j links to (0-based)."""
    n = len(links)
    x = np.ones(n) / n
    for _ in range(steps):
        y = np.full(n, (1 - d) / n)
        for j, out in enumerate(links):
            if out:
                for i in out:
                    y[i] += d * x[j] / len(out)
            else:                          # dead end: jump anywhere
                y += d * x[j] / n
        x = y
    return x

dead_end = [[1, 2], [2], [], [0, 2], [3, 5], [3, 4]]
print(np.round(pagerank(dead_end), 3))

The result, about (0.143, 0.130, 0.314, 0.172, 0.121, 0.121), matches the app. C, the dead end, ranks highest: three pages link to it, and since the surfer leaves it by a random jump, its rank is shared out evenly instead of being passed to any one page.

## Chapter 36: Population models

### Exercise 6

For the loggerhead model, project the population for 100 years from 1000 eggs and hatchlings and nothing else. Plot the total population on a logarithmic scale. After the first few decades, what is its slope, and how does it relate to $\lambda$? How many years does it take for the proportion in each stage to come within 1% of the stable distribution?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([[0, 0, 0, 0, 127, 4, 80],
              [0.6747, 0.7370, 0, 0, 0, 0, 0],
              [0, 0.0486, 0.6610, 0, 0, 0, 0],
              [0, 0, 0.0147, 0.6907, 0, 0, 0],
              [0, 0, 0, 0.0518, 0, 0, 0],
              [0, 0, 0, 0, 0.8091, 0, 0],
              [0, 0, 0, 0, 0, 0.8091, 0.8089]])
values, vectors = np.linalg.eig(A)
k = np.argmax(values.real)
stable = vectors[:, k].real / vectors[:, k].real.sum()

x = np.array([1000.0, 0, 0, 0, 0, 0, 0])
totals, gaps = [], []
for year in range(100):
    x = A @ x
    totals.append(x.sum())
    gaps.append(np.max(np.abs(x / x.sum() - stable)))
plt.figure(figsize=(6, 3.5))
plt.semilogy(range(1, 101), totals)
plt.xlabel("year"); plt.ylabel("total population")
plt.show()
print(np.log(totals[-1] / totals[-11]) / 10, np.log(values[k].real))
print(next(y + 1 for y in range(100) if all(g < 0.01 for g in gaps[y:])))

After the first two decades the total falls along a straight line on the log scale, with slope $\ln\lambda = \ln 0.945 \approx -0.0565$ per year. The proportions come within 1% of the stable distribution, and stay there, after 20 years. Early on, the total behaves quite differently, because the eggs take years to grow into breeders. MATLAB code follows the same steps with `semilogy`.

## Chapter 37: Data: regression and dimensionality reduction

### Exercise 6

Fit sepal length on all three other measurements. Then refit using only the 50 *setosa* flowers. Compare the coefficients and their signs, and the condition numbers of the two design matrices. What does this suggest about fitting one model to data from several groups?

In [ ]:
import numpy as np

data = np.genfromtxt(DATA + "iris.csv", delimiter=",", skip_header=1, dtype=str)
X = data[:, :4].astype(float)
species = data[:, 4]
for label, rows in [("all flowers", np.ones(150, bool)), ("setosa only", species == "setosa")]:
    A = np.column_stack([np.ones(rows.sum()), X[rows, 1:4]])
    beta, *_ = np.linalg.lstsq(A, X[rows, 0], rcond=None)
    print(f"{label}: {np.round(beta, 3)}, cond = {np.linalg.cond(A):.1f}")

For all 150 flowers, the coefficient of petal width is negative ($-0.556$); for *setosa* alone, all three coefficients are positive. The condition numbers are similar (about 50), so this is not just collinearity: the pooled model mixes three species, and the relationships *within* each species differ from the pattern *between* species. This is a version of **Simpson's paradox**. Fitting one model to data from distinct groups can give coefficients that describe none of them; including the group as a variable, or fitting each group separately, is safer.

## Chapter 38: Physics: rotations, inertia and quantum states

### Exercise 6

A quantum computer applies the **Hadamard gate** $\mathbf{H} = \tfrac{1}{\sqrt{2}}\begin{pmatrix}1 & 1 \\ 1 & -1\end{pmatrix}$ to a qubit. Check that $\mathbf{H}$ is symmetric and orthogonal, and that $\mathbf{H}^2 = \mathbf{I}$. Starting from $(1, 0)^{\mathsf{T}}$, apply $\mathbf{H}$, then a phase gate $\begin{pmatrix}1 & 0 \\ 0 & e^{i\phi}\end{pmatrix}$, then $\mathbf{H}$ again, and plot the probability of measuring $(1, 0)^{\mathsf{T}}$ as $\phi$ runs from 0 to $2\pi$. (This is the core of a quantum interferometer.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
print(np.allclose(H, H.T), np.allclose(H.T @ H, np.eye(2)), np.allclose(H @ H, np.eye(2)))
phis = np.linspace(0, 2 * np.pi, 200)
p0 = []
for phi in phis:
    phase = np.diag([1, np.exp(1j * phi)])
    psi = H @ phase @ H @ np.array([1, 0])
    p0.append(abs(psi[0])**2)
plt.figure(figsize=(6, 3))
plt.plot(phis, p0)
plt.xlabel("φ"); plt.ylabel("P(measure (1, 0))")
plt.show()

$\mathbf{H}$ is symmetric and orthogonal, and so $\mathbf{H}^2 = \mathbf{H}^{\mathsf{T}}\mathbf{H} = \mathbf{I}$. The probability is $\cos^2(\phi/2)$: 1 when $\phi = 0$ (the two Hadamard gates undo each other), 0 when $\phi = \pi$, and in between in a smooth interference pattern. The first $\mathbf{H}$ splits the state into an equal combination of the two basis states, the phase shifts one relative to the other, and the second $\mathbf{H}$ recombines them, so that they add or cancel depending on $\phi$, like light in an interferometer.

## Chapter 39: Cryptography: the Hill cipher

### Exercise 6

The ciphertext `FRFMKCFRJGBF` was encrypted with an unknown $2 \times 2$ Hill key, and you know the message begins with ATTA. Find the key and decrypt the whole message.

In [ ]:
import numpy as np

def to_nums(text):
    return [ord(ch) - ord("A") for ch in text]

def inverse_mod26(K):
    det = round(np.linalg.det(K)) % 26
    adj = np.array([[K[1, 1], -K[0, 1]], [-K[1, 0], K[0, 0]]])
    return pow(det, -1, 26) * adj % 26

cipher = "FRFMKCFRJGBF"
P = np.array(to_nums("ATTA")).reshape(2, 2).T     # known plaintext pairs as columns
C = np.array(to_nums(cipher[:4])).reshape(2, 2).T
K = C @ inverse_mod26(P) % 26
print(K)
pairs = np.array(to_nums(cipher)).reshape(-1, 2).T
print("".join(chr(int(n) + 65) for n in (inverse_mod26(K) @ pairs % 26).T.ravel()))

The known pairs AT and TA give $\mathbf{P} = \begin{pmatrix}0 & 19 \\ 19 & 0\end{pmatrix}$, with determinant $-361 \equiv 3$, which is invertible. The key is $\begin{pmatrix}3 & 3 \\ 2 & 5\end{pmatrix}$, and the message is ATTACKATDAWN. In MATLAB, the functions `hill` and `inv26` from the chapter do the same in three lines.

## Chapter 40: Vibrations and normal modes

### Exercise 7

For $n$ equal masses $m$ joined by $n + 1$ equal springs $k$ between two walls, the natural frequencies are
$$
\omega_j = 2\sqrt{k/m}\,\sin\frac{j\pi}{2(n + 1)},
\qquad j = 1, \dots, n.
$$
Check this numerically for $n = 10$, $m = 1$ and $k = 100$. Plot the first three mode shapes for $n = 30$. What shapes are they, and what physical object does a long chain of masses and springs approximate?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

m, k, n = 1, 100, 10
K = k * (2 * np.eye(n) - np.eye(n, k=1) - np.eye(n, k=-1))
omega = np.sqrt(np.linalg.eigvalsh(K) / m)       # M = m I, so an ordinary eigenproblem
exact = 2 * np.sqrt(k / m) * np.sin(np.arange(1, n + 1) * np.pi / (2 * (n + 1)))
print("agree:", np.allclose(omega, exact))

n = 30                                           # mode shapes for a longer chain
K = 2 * np.eye(n) - np.eye(n, k=1) - np.eye(n, k=-1)
lam, V = np.linalg.eigh(K)
plt.figure(figsize=(6, 3))
for j in range(3):
    plt.plot(range(1, n + 1), V[:, j], "o-", ms=3, label=f"mode {j + 1}")
plt.xlabel("mass number"); plt.ylabel("displacement"); plt.legend()
plt.show()

The mode shapes are samples of sine curves: half a wave for mode 1, a whole wave for mode 2, one and a half for mode 3, each zero at the walls (the signs may be flipped). A long chain of masses and springs approximates a stretched string, such as a guitar string, whose modes are exactly these sines, with frequencies in the ratio $1 : 2 : 3$ for the low modes. The matrix is the finite difference matrix of [Chapter 33](https://trgoodale.github.io/matrix-book/part-6/33-finite-differences.html).

## Chapter 41: Networks and graphs

### Exercise 7

Make a random network with two hidden communities of 50 nodes each: join each pair of nodes in the same community with probability 0.3, and each pair in different communities with probability 0.05. Find the Fiedler vector, split the nodes by its signs, and count how many nodes end up in the right group. Repeat with a between-community probability of 0.15. What happens, and why?

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
n = 100
group = np.repeat([0, 1], 50)
same = group[:, None] == group[None, :]
for p_out in [0.05, 0.15]:
    P = np.where(same, 0.3, p_out)
    A = np.triu(rng.random((n, n)) < P, 1).astype(float)
    A = A + A.T                                  # each pair joined at most once
    L = np.diag(A.sum(axis=1)) - A
    lam, V = np.linalg.eigh(L)
    guess = V[:, 1] > 0
    correct = max(np.sum(guess == group), np.sum(guess != group))   # the labels may be swapped
    print(f"p_out = {p_out}: lambda2 = {lam[1]:.2f}, {correct} of {n} correct")

The results depend on the random numbers, so MATLAB's differ from Python's. With between-community probability 0.05, the Fiedler vector puts all, or almost all, of the 100 nodes in the right group. With 0.15, only about 60 to 90 are right, depending on the random network (half would be right by pure chance). Each node then has about 15 neighbours in its own community and 7.5 in the other, so the communities are much less distinct: many nodes have almost as many links across as within, and random variation decides which side they fall. As the two probabilities approach each other, the hidden communities become impossible to detect by any method.

## Chapter 42: Economics: input–output models

### Exercise 7

In Example 4, manufacturers cut their direct emissions by 20%. Find the new footprints of all three products. Why does the footprint of food fall, although farms emit as much as before? Which product's footprint falls by the largest fraction?

In [ ]:
import numpy as np

A = np.array([[0.10, 0.05, 0.02], [0.20, 0.35, 0.10], [0.10, 0.20, 0.25]])
L = np.linalg.inv(np.eye(3) - A)
e = np.array([1.0, 0.6, 0.1])
e_new = e * np.array([1, 0.8, 1])                # manufacturing cuts by 20%
f, f_new = e @ L, e_new @ L
print(np.round(f, 3))
print(np.round(f_new, 3))
print(np.round(100 * (f - f_new) / f, 1), "% lower")

The footprints fall from about $(1.40, 1.13, 0.32)$ to $(1.35, 0.93, 0.29)$ kg per £1. Food's footprint falls, although farms emit as much as before, because farming buys manufactured goods: 20p of them per £1 of output, such as fertiliser and machinery, whose emissions are part of food's supply chain. Manufactured goods fall most, by 17%, a little less than 20% because part of their footprint comes from their agricultural and service inputs. Services fall by 9%, through the manufactured goods they buy. Cutting emissions anywhere in a supply chain lowers the footprint of everything downstream.

## Part VI: Applications review

### Exercise 11

Five web pages link as follows: A links to B; B links to A and C; C links to A; D links to C and E; E links to D. Find the PageRank with damping $d = 0.85$ by the power method, counting the iterations needed, and check it by solving a linear system. Explain why D and E rank lowest, even though they link to each other.

*Uses [Chapter 35](https://trgoodale.github.io/matrix-book/part-6/35-markov-pagerank.html), [Chapter 29](https://trgoodale.github.io/matrix-book/part-5/29-eigenvalue-algorithms.html) and [Chapter 26](https://trgoodale.github.io/matrix-book/part-5/26-cost-sparse.html).*

In [ ]:
import numpy as np

links = {0: [1], 1: [0, 2], 2: [0], 3: [2, 4], 4: [3]}   # pages A to E as 0 to 4
n, d = 5, 0.85
M = np.zeros((n, n))
for j, targets in links.items():
    M[targets, j] = 1 / len(targets)

x = np.ones(n) / n
for k in range(1, 1001):
    x_new = d * M @ x + (1 - d) / n
    if np.abs(x_new - x).sum() < 1e-10:
        break
    x = x_new
x = x_new
print(k, "iterations")
print(np.round(x, 4))
print(np.round(np.linalg.solve(np.eye(n) - d * M, (1 - d) / n * np.ones(n)), 4))

The ranks, A to E, are 0.333, 0.313, 0.200, 0.087 and 0.067, and the linear system agrees. The power method took 43 steps, in line with the error shrinking by a factor of about 0.85 per step. D and E rank lowest because no page outside the pair links to them: all they receive is the random jumps and what they pass to each other, and D leaks half of its rank to C.